<a href="https://colab.research.google.com/github/joshicallyc/Python/blob/main/learning/python-refresher/python_refresher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐍 Python refresher: back up to speed in one evening

<a href="https://colab.research.google.com/github/joshicallyc/Python/blob/main/learning/python-refresher/python_refresher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook reviews the Python you learned this summer, using DNA instead of toy examples, and ends by
running **your own DNA translator** so you see its bug before Thursday's big night.

**About 60–75 minutes.** Short on time? Do **Parts 0–3 and Part 7** (about 40 minutes).

**How it works**
- Click a gray code cell and press **Shift + Enter** to run it. Run cells from top to bottom.
- In each **✏️ Your turn** cell, replace `None` (or follow the `# YOUR CODE` comment), then run it:
  - ✅ = correct
  - ❌ = not yet, and it shows what it expected
  - ⬜ = you haven't tried it yet
- Stuck for 5 minutes? Open **💡 Hint**. Still stuck? Open **🔑 Answer**, then type it in yourself.
- Your edits don't change anything on GitHub, so experiment freely. To keep your answers:
  **File → Save a copy in Drive**.

## Part 0 · Setup (2 min)
This cell downloads your GitHub project into Colab and creates `check()`, which marks your answers.

In [66]:
# Run me first! (click here, then press Shift + Enter)
!test -d /content/Python || git clone -q https://github.com/joshicallyc/Python.git /content/Python

def check(name, got, expected):
    """Prints ✅ if your answer matches, ❌ (with details) if not, ⬜ if you haven't tried yet."""
    if got is None or (got in ("", []) and expected not in ("", [])):
        print(f"⬜ {name}: not done yet")
    elif isinstance(expected, float) and isinstance(got, (int, float)) and abs(got - expected) < 1e-9:
        print(f"✅ {name}")
    elif got == expected:
        print(f"✅ {name}")
    else:
        print(f"❌ {name}: expected {expected!r}, but got {got!r}")

print("Ready! Files in your dna-translator folder:")
!ls /content/Python/dna-translator

Ready! Files in your CLI_project folder:
amino_acid_sequence.txt  fasta.txt	      mRNA.txt
codon.txt		 humandna.fasta.html  reader.py


## Part 1 · Strings (10 min)
A DNA sequence is just a **string**. These are the tools you'll use constantly:

| Code | What it does | Result when `dna = "ATGCGT"` |
|---|---|---|
| `len(dna)` | length | `6` |
| `dna[0]` | first letter (Python counts from **0**) | `'A'` |
| `dna[0:3]` | letters 0, 1, 2 (the stop number is *not* included) | `'ATG'` |
| `dna[-1]`, `dna[-3:]` | counting from the end | `'T'`, `'CGT'` |
| `dna.count("G")` | how many times `"G"` appears | `2` |
| `dna.upper()`, `dna.lower()` | change case | `'ATGCGT'`, `'atgcgt'` |
| `dna.replace("T", "U")` | swap letters | `'AUGCGU'` |
| `"GCG" in dna` | is it inside? | `True` |

In [67]:
# Example: run it and compare with the table above
dna = "ATGCGT"
print(len(dna), dna[0], dna[0:3], dna[-3:], dna.count("G"))
print(dna.replace("T", "U"), "GCG" in dna)

6 A ATG CGT 2
AUGCGU True


### ✏️ Your turn

In [68]:
gene = "atggccattgtaatgggccgctga"   # lowercase, like the sequence lines in your fasta.txt

# 1a. Make it uppercase
gene_upper = gene.upper()
check("1a uppercase", gene_upper, "ATGGCCATTGTAATGGGCCGCTGA")

# 1b. The first codon (first 3 letters) of gene_upper
first_codon = gene_upper[:3]
check("1b first codon", first_codon, "ATG")

# 1c. The LAST codon (last 3 letters) of gene_upper
last_codon = gene_upper[-3:]
check("1c last codon", last_codon, "TGA")

# 1d. GC content = (number of G + number of C) / length
gc = (gene_upper.count("G") + gene_upper.count("C"))/(len(gene_upper))
check("1d GC content", gc, 0.5416666666666666)

# 1e. The mRNA version: every T becomes U
mrna = gene_upper.replace("T", "U")
check("1e mRNA", mrna, "AUGGCCAUUGUAAUGGGCCGCUGA")

✅ 1a uppercase
✅ 1b first codon
✅ 1c last codon
✅ 1d GC content
✅ 1e mRNA


<details><summary>💡 Hint</summary>

Strings have *methods* you call with a dot: `.upper()`, `.count("G")`, `.replace("T", "U")`. Slices look like `s[start:stop]`. For the last three letters, start at `-3` and leave the stop empty: `s[-3:]`.

</details>
<details><summary>🔑 Answer (try the hint first!)</summary>

```python
gene_upper = gene.upper()
first_codon = gene_upper[0:3]
last_codon = gene_upper[-3:]
gc = (gene_upper.count("G") + gene_upper.count("C")) / len(gene_upper)
mrna = gene_upper.replace("T", "U")
```

</details>

## Part 2 · Loops: the key to your translator's bug (10 min)
There are two very different ways to loop over a string. Run both and compare what gets printed.

In [69]:
dna = "ATGCGT"

print("Loop 1: for letter in dna")
for letter in dna:              # one LETTER at a time
    print("  ", letter)

print("Loop 2: for i in range(0, len(dna), 3)")
for i in range(0, len(dna), 3): # i = 0, 3, 6, ... (steps of 3)
    print("  ", i, dna[i:i + 3])  # a 3-letter slice = one codon

Loop 1: for letter in dna
   A
   T
   G
   C
   G
   T
Loop 2: for i in range(0, len(dna), 3)
   0 ATG
   3 CGT


👀 Loop 1 gives single letters, and Loop 2 gives codons. Remember this difference: it's the bug you'll
fix on Thursday.

### ✏️ Your turn

In [70]:
gene_upper = "ATGGCCATTGTAATGGGCCGCTGA"   # (same as Part 1)

# 2a. Build a list of all the codons in gene_upper.
codons = []
for i in range(0, len(gene_upper), 3):
  codons.append(gene_upper[i:i+3])
# YOUR CODE: loop with range(0, len(gene_upper), 3) and use codons.append(...) to add each 3-letter slice

check("2a codons", codons, ["ATG", "GCC", "ATT", "GTA", "ATG", "GGC", "CGC", "TGA"])

# 2b. How many of those codons are "ATG"? (Lists have a .count() method too.)
atg_count = codons.count("ATG")
check("2b ATG count", atg_count, 2)

✅ 2a codons
✅ 2b ATG count


<details><summary>💡 Hint</summary>

Copy Loop 2 from the example, but loop over `gene_upper`, and instead of printing, write `codons.append(gene_upper[i:i + 3])` inside the loop (indented!). For 2b: `codons.count("ATG")`.

</details>
<details><summary>🔑 Answer (try the hint first!)</summary>

```python
codons = []
for i in range(0, len(gene_upper), 3):
    codons.append(gene_upper[i:i + 3])

atg_count = codons.count("ATG")
```

</details>

## Part 3 · if / elif / else, and `==` vs `in` (8 min)
- `==` compares with **one** value: `codon == "ATG"`
- `in` checks whether something is in a **group**: `codon in ("TAA", "TAG", "TGA")`

Two classic traps (both show up in your translator's code). Run this and look at the third line.

In [71]:
codon = "GCC"
print(codon == ("TAA", "TAG", "TGA"))   # False: a string is never equal to a whole group
print(codon in ("TAA", "TAG", "TGA"))   # False: correct, GCC isn't a stop codon
print(bool(codon == "TAA" or "TAG"))    # True?! Python reads this as (codon == "TAA") or ("TAG"),
                                        #        and a non-empty string like "TAG" counts as True

False
False
True


### ✏️ Your turn

In [72]:
codons = ["ATG", "GCC", "ATT", "GTA", "ATG", "GGC", "CGC", "TGA"]   # (same as your answer to 2a)

# 3a. Label each codon "start" (ATG), "stop" (TAA, TAG or TGA), or "other"
labels = []
for codon in codons:
    # YOUR CODE: use if / elif / else, then labels.append(...) the right word
    if codon == "ATG":
      labels.append("start")
    elif codon in ("TAA", "TAG", "TGA"):
      labels.append("stop")
    else:
      labels.append("other")

check("3a labels", labels, ["start", "other", "other", "other", "start", "other", "other", "stop"])

✅ 3a labels


<details><summary>💡 Hint</summary>

Replace `pass` with three branches: `if codon == "ATG":` → `labels.append("start")`, `elif codon in ("TAA", "TAG", "TGA"):` → `"stop"`, `else:` → `"other"`. Watch the colons and the indentation.

</details>
<details><summary>🔑 Answer (try the hint first!)</summary>

```python
for codon in codons:
    if codon == "ATG":
        labels.append("start")
    elif codon in ("TAA", "TAG", "TGA"):
        labels.append("stop")
    else:
        labels.append("other")
```

</details>

## Part 4 · Dictionaries: a codon table (10 min)
A **dictionary** maps *keys* to *values*, which is exactly what a codon table is.
- Look something up: `table["ATG"]` gives `"Met"`
- Look up a key that might be missing: `table.get(codon, "?")` gives `"?"` instead of crashing
- Join a list of strings: `"-".join(["Met", "Ala"])` gives `"Met-Ala"`

(Real codon tables often use RNA letters, like `AUG`. The idea is the same.)

In [73]:
table = {"ATG": "Met", "TGA": "Stop"}
print(table["ATG"])
print(table.get("CCC", "?"))       # CCC isn't in this small table, so we get "?"
print("-".join(["Met", "Ala", "Gly"]))

Met
?
Met-Ala-Gly


### ✏️ Your turn

In [74]:
mini_table = {
    "ATG": "Met", "GCC": "Ala", "ATT": "Ile", "GTA": "Val",
    "GGC": "Gly", "CGC": "Arg", "TGA": "Stop",
}
codons = ["ATG", "GCC", "ATT", "GTA", "ATG", "GGC", "CGC", "TGA"]

# 4a. Look up the amino acid for "GTA"
aa = mini_table["GTA"]

check("4a lookup", aa, "Val")

# 4b. Translate the codons in order. Stop when you reach "Stop" (don't add it).
protein = []
for codon in codons:
    # YOUR CODE: look up the amino acid; if it's "Stop", use `break`; otherwise append it
    amino_acid = mini_table[codon]
    if amino_acid == "Stop":
      break
    else:
      protein.append(amino_acid)
check("4b protein", protein, ["Met", "Ala", "Ile", "Val", "Met", "Gly", "Arg"])

# 4c. Join the protein list into one string with dashes between the names
protein_str = "-".join(protein)
check("4c joined", protein_str, "Met-Ala-Ile-Val-Met-Gly-Arg")

✅ 4a lookup
✅ 4b protein
✅ 4c joined


<details><summary>💡 Hint</summary>

4a: `mini_table["GTA"]`. 4b: inside the loop, `amino_acid = mini_table[codon]`, then `if amino_acid == "Stop": break`, otherwise `protein.append(amino_acid)`. 4c: `"-".join(protein)`.

</details>
<details><summary>🔑 Answer (try the hint first!)</summary>

```python
aa = mini_table["GTA"]

protein = []
for codon in codons:
    amino_acid = mini_table[codon]
    if amino_acid == "Stop":
        break
    protein.append(amino_acid)

protein_str = "-".join(protein)
```

</details>

## Part 5 · Functions (8 min)
A **function** gives a name to a few steps so you can reuse them. `return` hands a value back, so you
can store it in a variable. (`print` only shows it on screen.)

In [75]:
def gc_content(seq):
    seq = seq.upper()
    return (seq.count("G") + seq.count("C")) / len(seq)

print(gc_content("atgc"))       # 0.5
print(gc_content("GGGGCCCCAT")) # 0.8

0.5
0.8


### ✏️ Your turn

In [79]:
# 5a. transcribe(dna) should return the uppercase mRNA (every T becomes U).
#     (This assumes the coding strand. Keep that in mind for roadmap item P0-08!)
def transcribe(dna):
    return(dna.upper().replace("T", "U"))

check("5a transcribe", transcribe("atgc"), "AUGC")

# 5b. split_codons(seq) should return the list of 3-letter codons (reuse your Part 2 loop!)
def split_codons(seq):
    result = []
    for i in range(0, len(seq), 3):
      result.append(seq[i:i+3])
    return result

check("5b split_codons", split_codons("ATGGCCTGA"), ["ATG", "GCC", "TGA"])

✅ 5a transcribe
✅ 5b split_codons


<details><summary>💡 Hint</summary>

5a: one line, `return dna.upper().replace("T", "U")` (you can chain methods). 5b: your Part 2 loop, but use `seq` and `result` instead of `gene_upper` and `codons`.

</details>
<details><summary>🔑 Answer (try the hint first!)</summary>

```python
def transcribe(dna):
    return dna.upper().replace("T", "U")

def split_codons(seq):
    result = []
    for i in range(0, len(seq), 3):
        result.append(seq[i:i + 3])
    return result
```

</details>

## Part 6 · Reading your FASTA file (10 min)
A FASTA file has a **header line** that starts with `>`, followed by the sequence split across **many
lines**. `with open(...)` opens the file and closes it automatically when you're done. Run this to look
at the first two lines of your own file:

In [81]:
path = "/content/Python/dna-translator/fasta.txt"
with open(path) as f:
    lines = f.readlines()      # a list containing every line of the file

print(len(lines), "lines in the file")
print(repr(lines[0]))          # repr() shows hidden characters, like the \n (newline) at the end
print(repr(lines[1]))

12 lines in the file
'>BTBSCRYR\n'
'tgcaccaaacatgtctaaagctggaaccaaaattactttctttgaagacaaaaactttca\n'


Notice the `\n` at the end of each line. That's why you'll use `.strip()`.

### ✏️ Your turn

In [82]:
path = "/content/Python/dna-translator/fasta.txt"

header = None
sequence = ""
with open(path) as f:
    for line in f:
        line = line.strip()
        if line.startswith(">"):
          header = line
        else:
          sequence += line.upper()

             # removes the \n and spaces at both ends
        # YOUR CODE:
        #   if the line starts with ">", save it in `header`
        #   otherwise, add line.upper() to the end of `sequence`
        pass

check("6a header", header, ">BTBSCRYR")
check("6b sequence length", len(sequence) if sequence else None, 620)
check("6c first 12 letters", sequence[:12], "TGCACCAAACAT")

✅ 6a header
✅ 6b sequence length
✅ 6c first 12 letters


<details><summary>💡 Hint</summary>

Replace `pass` with `if line.startswith(">"):` → `header = line`, and `else:` → `sequence += line.upper()` (`+=` adds to the end of a string).

</details>
<details><summary>🔑 Answer (try the hint first!)</summary>

```python
if line.startswith(">"):
            header = line
        else:
            sequence += line.upper()
```

</details>

💡 **Why this matters:** your translator handles the file **one line at a time**, so any codon that
crosses a line break gets cut in half. Joining all the lines first, as you just did, fixes that. That's
roadmap item `P0-07`.

## Part 7 · Run YOUR translator and see the bug (10 min)
This is roadmap item **`P0-01`**. `reader.py` has since been rewritten (see the
[translator's README](https://github.com/joshicallyc/Python/tree/main/dna-translator)), but git keeps every
version, so the first cell pulls out your **first version** as `reader_v1.py`. Then it runs it on your FASTA
file the same way you'd run it in a terminal (the `!` means "run this as a terminal command").

In [ ]:
%cd /content/Python/dna-translator
!git show cdade1a:CLI_project/reader.py > reader_v1.py
!python3 reader_v1.py -f fasta.txt
print("---- codon.txt contains: ----")
!cat codon.txt

You should see only the header, `>BTBSCRYR`, and **no amino acids**. That's the bug. Here's the part
of the code that causes it (with line numbers):

In [ ]:
!cat -n reader_v1.py | sed -n '17,47p'

Now try the loop from line 33 on a tiny piece of mRNA:

In [ ]:
output = "AUGGCC"          # a tiny piece of mRNA
for codon in output:       # the same loop as line 33 of reader_v1.py
    print(repr(codon))

### ✏️ Your turn: three questions (no code)
Write your answers on your **Coding Now** page in Notion, under **"Questions & notes for Claude"**.
On Thursday's big night, you'll fix the code together with Claude.

1. On **line 33**, what does `codon` hold each time through the loop? Why can it never equal `'AUG'`?
2. **Lines 27–28** *do* make real 3-letter codons. What happens to each one? (Hint: what is `codon`
   after the loop has finished?)
3. **Lines 43 and 45** use `==` with a group in parentheses. Using Part 3, what do they always give?

**Don't fix it tonight.** Figuring out *why* is the hard part, and you just did it.

## 🎁 Bonus · Your first plot (5 min)
How much of each base is in your sequence? `matplotlib` turns the counts into a bar chart.

In [ ]:
import matplotlib.pyplot as plt

# Read the sequence again (a shorter way of doing Part 6; you'll learn this style later)
path = "/content/Python/dna-translator/fasta.txt"
with open(path) as f:
    seq = "".join(line.strip().upper() for line in f if not line.startswith(">"))

bases = ["A", "C", "G", "T"]
counts = [seq.count(base) for base in bases]

plt.bar(bases, counts, color="#2a78d6")
plt.title("Bases in your FASTA sequence (BTBSCRYR)")
plt.xlabel("Base")
plt.ylabel("Count")
plt.gca().spines[["top", "right"]].set_visible(False)   # optional: a cleaner look
plt.show()

print(dict(zip(bases, counts)))
print("GC content:", round((seq.count("G") + seq.count("C")) / len(seq), 3))

## ✅ Done! What's next
1. On **Coding Now**, tick **`P0-01`** and write your three Part 7 answers under
   **"Questions & notes for Claude"**.
2. **Thursday's big night:** go to [claude.ai/code](https://claude.ai/code), choose the
   `joshicallyc/Python` repo, and type **start**.
3. Any part that felt shaky? Run its cells again later. Repetition is how syntax sticks.